# HarvestStat Data Analysis - Nigeria

In [ ]:
import pandas as pd
df = pd.read_excel('/Users/dlee/Downloads/Nigeria_Oct112023.xlsx')

In [ ]:
df

## Issue 1: Duplicates

In [ ]:
import requests, json
import pandas as pd
import numpy as np

host = 'https://fdw.fews.net'
auth = tuple(json.loads(open('token.json', "r").read()))
parameters = {
    'format': 'json',
    'country': 'Nigeria',
    'product': 'R011',
    'survey_type': 'crop:all'
}
endpoint = '/api/cropproductionindicatorvalue/'
response = requests.get(host + endpoint, auth=auth, params=parameters, proxies={})
response.raise_for_status()
df = pd.DataFrame.from_records(response.json())
rows = ['fnid','crop_production_system','season_year', 'product','indicator']
sub = df[df['collection_status'] == 'Published']
sub = sub[sub[rows].duplicated(keep=False)]
sub.pivot_table(index = ['fnid','season_year', 'product','indicator'], columns = ['source_document', 'publication_name'], values='value')

In [ ]:
df[['status_changed']].apply(pd.to_datetime).apply(lambda x: x.dt.strftime('%Y-%m-%d')).drop_duplicates()

In [ ]:
import requests, json
import pandas as pd
import numpy as np

host = 'https://fdw.fews.net'
auth = tuple(json.loads(open('token.json', "r").read()))
parameters = {
    'format': 'json',
    'country': 'Nigeria',
    'product': 'R011',
    'survey_type': 'crop:all'
}
endpoint = '/api/cropproductionindicatorvalue/'
response = requests.get(host + endpoint, auth=auth, params=parameters, proxies={})
response.raise_for_status()
df = pd.DataFrame.from_records(response.json())
rows = ['fnid','crop_production_system','season_year', 'product','indicator']
sub = df[df['collection_status'] == 'Published']
sub = sub[sub[rows].duplicated(keep=False)]
sub.pivot_table(index = ['fnid','season_year', 'product','indicator'], columns = ['source_document', 'publication_name'], values='value')

In [ ]:
print(df[['source_document','publication_name']].drop_duplicates().to_string())

In [ ]:
df[['fnid','admin_1']].drop_duplicates()

In [ ]:
df[
    # (df['admin_1'] == 'Abia') &
    (df['product']  == 'Maize (Corn)') &
    (df['indicator'] == 'Quantity Produced')
].pivot_table(index = ['season_year'], columns = ['source_document','publication_name'], values='value', aggfunc='count')

In [ ]:
df[
    # (df['admin_1'] == 'Borno') &
    (df['product']  == 'Maize (Corn)') &
    (df['indicator'] == 'Quantity Produced')
].pivot_table(index = ['season_year'], columns = ['source_document','publication_name'], values='value', aggfunc='sum')

In [ ]:
df[
    # (df['admin_1'] == 'Abia') &
    (df['product']  == 'Maize (Corn)') &
    (df['indicator'] == 'Quantity Produced') &
    (df['season_year'].isin(['Wet 2014', 'Wet 2015']))
].pivot_table(index = ['admin_1'], columns = ['season_year'], values='value', aggfunc='sum')

In [ ]:
# Load FAO-STAT National Production Data ------------- #
tmp = pd.read_csv('../data/crop/adm_fao_stat.csv', index_col=0)
data_fao = tmp[
    (tmp['cnt_name'] == 'Nigeria') &
    (tmp['cpc2_name'] == 'Maize (corn)') &
    (tmp['indicator'] == 'Production')
].set_index('year')['value']
# ---------------------------------------------------- #
# data = pd.concat([data_fao, data_fdw], axis=1, keys=['FAO','GSCD']).sort_index()
# data.reindex(np.array(range(1961,2024)))